# 🔍 VeriLens AI — Explainable Fake News Detection
## IEEE DataPort Hackathon · VeriLens Trust Index (VTI) v2.0

> *From raw text to trustworthy verdict — powered by DistilBERT + XGBoost + SHAP*

---

| Component | Technology | Purpose |
|---|---|---|
| Language Model | DistilBERT (prototype similarity) | Semantic fake-probability |
| Classifier | XGBoost (300 trees) | Hybrid feature classification |
| Explainability | SHAP TreeExplainer | Per-prediction feature attribution |
| Linguistic Analysis | 26 handcrafted features | Stylometric signals |
| Speaker Memory | LIAR credibility profiles | Historical speaker reliability |
| Trust Score | VeriLens Trust Index (VTI) | 6-component weighted verdict |
| OCR | EasyOCR | Screenshot / image input |
| Backend | FastAPI | REST API (`/predict`, `/ocr`, `/health`) |
| Frontend | Streamlit | Interactive analysis dashboard |

### Pipeline
```
User Input (Text / Screenshot)
        ↓
   OCR (if image)
        ↓
  Text Cleaning
        ↓
DistilBERT Embeddings ──────────────────────────┐
        ↓                                        │
Linguistic Features (26) → XGBoost → SHAP       │
        ↓                                        ↓
   Speaker Memory              Model Confidence Score
        ↓                               ↓
        └───────── VeriLens Trust Index (VTI) ───┘
                           ↓
              Interactive Dashboard / API
```


In [2]:
# ── Install all dependencies ────────────────────────────────
!pip install xgboost shap transformers torch plotly easyocr textstat fastapi uvicorn streamlit -q
print("✅ Dependencies installed")


✅ Dependencies installed


## ⚙️ Module 1 — Configuration

All weights, thresholds, and paths in one place. No magic numbers elsewhere.

In [3]:
# ══════════════════════════════════════════════════════════════
#  VeriLens AI — config.py (embedded)
# ══════════════════════════════════════════════════════════════

# ── VeriLens Trust Index component weights (must sum to 1.0) ─
VTI_WEIGHTS = {
    'MODEL':       0.35,   # XGBoost + DistilBERT weighted prediction
    'SPEAKER':     0.20,   # Speaker credibility from historical memory
    'LANGUAGE':    0.15,   # Language neutrality (emotional / sensational)
    'CLICKBAIT':   0.10,   # Clickbait pattern density
    'CONSISTENCY': 0.10,   # Evidence richness (quotes, numbers, citations)
    'METADATA':    0.10,   # Structural quality (readability, formatting)
}
assert abs(sum(VTI_WEIGHTS.values()) - 1.0) < 1e-9, "❌ VTI_WEIGHTS must sum to 1.0"

# ── Verdict thresholds ────────────────────────────────────────
VTI_THRESHOLDS = {
    'HIGHLY_TRUSTWORTHY': 90,
    'MOSTLY_RELIABLE':    70,
    'NEEDS_VERIFICATION': 50,
    'SUSPICIOUS':         30,
}

# ── Dataset paths (update for your Kaggle environment) ───────
DATASET_PATHS = {
    'liar_train': '/kaggle/input/datasets/doanquanvietnamca/liar-dataset/train.tsv',
    'liar_valid': '/kaggle/input/datasets/doanquanvietnamca/liar-dataset/valid.tsv',
    'liar_test':  '/kaggle/input/datasets/doanquanvietnamca/liar-dataset/test.tsv',
    'fnn_base':   '/kaggle/input/datasets/mdepak/fakenewsnet/',
}

# ── Model hyperparameters ─────────────────────────────────────
MODEL_CONFIG = {
    'bert_name':      'distilbert-base-uncased',
    'bert_max_len':   128,
    'xgb_estimators': 300,
    'xgb_depth':      6,
    'xgb_lr':         0.05,
    'xgb_subsample':  0.8,
    'xgb_colsample':  0.8,
    'test_size':      0.20,
    'random_state':   42,
}

# ── Output paths ──────────────────────────────────────────────
OUTPUT_DIR         = '/kaggle/working/'
MODEL_SAVE_PATH    = OUTPUT_DIR + 'verilens_xgb_model.joblib'
SCALER_SAVE_PATH   = OUTPUT_DIR + 'verilens_scaler.joblib'
SPEAKER_SAVE_PATH  = OUTPUT_DIR + 'speaker_memory.json'
METRICS_SAVE_PATH  = OUTPUT_DIR + 'verilens_metrics.json'

print("✅ VeriLens AI Configuration loaded")
print(f"\n  VTI Weights  : {VTI_WEIGHTS}")
print(f"  Thresholds   : {VTI_THRESHOLDS}")


✅ VeriLens AI Configuration loaded

  VTI Weights  : {'MODEL': 0.35, 'SPEAKER': 0.2, 'LANGUAGE': 0.15, 'CLICKBAIT': 0.1, 'CONSISTENCY': 0.1, 'METADATA': 0.1}
  Thresholds   : {'HIGHLY_TRUSTWORTHY': 90, 'MOSTLY_RELIABLE': 70, 'NEEDS_VERIFICATION': 50, 'SUSPICIOUS': 30}


## 📦 Module 2 — Imports & Logging Setup

In [4]:
import pandas as pd
import numpy as np
import re, math, nltk, warnings, json, joblib, logging, os, shutil
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix,
                              ConfusionMatrixDisplay, roc_curve, classification_report)
from xgboost import XGBClassifier
import shap
import torch
from transformers import AutoTokenizer, AutoModel
from tqdm.auto import tqdm
import plotly.express as px
import plotly.graph_objects as go
warnings.filterwarnings('ignore')

# ── Logging Setup ────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-8s | %(message)s',
    datefmt='%H:%M:%S'
)
logger = logging.getLogger('VeriLens')
logger.setLevel(logging.INFO)

# ── NLTK Downloads ───────────────────────────────────────────
for _r in ['punkt', 'stopwords', 'averaged_perceptron_tagger', 'punkt_tab']:
    nltk.download(_r, quiet=True)
from nltk.corpus import stopwords
from nltk.tokenize import sent_tokenize, word_tokenize
STOP = set(stopwords.words('english'))

# ── Textstat for readability ─────────────────────────────────
try:
    import textstat
    TEXTSTAT_AVAILABLE = True
    logger.info("textstat loaded ✓")
except ImportError:
    TEXTSTAT_AVAILABLE = False
    logger.warning("textstat not available — readability defaults to 0.5")

# ── Project folder structure ──────────────────────────────────
for _d in ['app/backend', 'app/frontend', 'app/ai', 'app/ocr',
           'trained_models', 'docs', 'tests']:
    os.makedirs(_d, exist_ok=True)
logger.info("Project directories created")

logger.info("✅ All imports successful")
print("✅ All imports successful")


13:39:24 | INFO     | textstat loaded ✓
13:39:24 | INFO     | Project directories created
13:39:24 | INFO     | ✅ All imports successful


✅ All imports successful


## 📂 Module 3 — Dataset Loading

Loads LIAR (political fact-checks) and FakeNewsNet (PolitiFact + BuzzFeed articles).

In [5]:
# ── LIAR Dataset (TSV) ───────────────────────────────────────
LIAR_COLS = [
    'id', 'label', 'statement', 'subject', 'speaker',
    'job_title', 'state', 'party',
    'barely_true_ct', 'false_ct', 'half_true_ct',
    'mostly_true_ct', 'pants_fire_ct', 'context'
]

train_df = pd.read_csv(DATASET_PATHS['liar_train'], sep='\t', header=None, names=LIAR_COLS)
val_df   = pd.read_csv(DATASET_PATHS['liar_valid'], sep='\t', header=None, names=LIAR_COLS)
test_df  = pd.read_csv(DATASET_PATHS['liar_test'],  sep='\t', header=None, names=LIAR_COLS)

liar_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
logger.info(f"LIAR dataset loaded: {len(liar_df)} rows")
print(f"✅ LIAR loaded: {len(liar_df)} rows")

# ── FakeNewsNet CSVs ─────────────────────────────────────────
_BASE = DATASET_PATHS['fnn_base']
pf_fake = pd.read_csv(_BASE + 'PolitiFact_fake_news_content.csv', on_bad_lines='skip')
pf_real = pd.read_csv(_BASE + 'PolitiFact_real_news_content.csv', on_bad_lines='skip')
bf_real = pd.read_csv(_BASE + 'BuzzFeed_real_news_content.csv',  on_bad_lines='skip')

pf_fake['binary_label'] = 0
pf_real['binary_label'] = 1
bf_real['binary_label'] = 1

logger.info(f"FakeNewsNet loaded: pf_fake={pf_fake.shape}, pf_real={pf_real.shape}, bf_real={bf_real.shape}")
print(f"PolitiFact fake: {pf_fake.shape} | PolitiFact real: {pf_real.shape} | BuzzFeed real: {bf_real.shape}")
print(f"\nPolitiFact fake columns: {pf_fake.columns.tolist()}")


13:39:40 | INFO     | LIAR dataset loaded: 12791 rows
13:39:40 | INFO     | FakeNewsNet loaded: pf_fake=(120, 13), pf_real=(120, 13), bf_real=(91, 13)


✅ LIAR loaded: 12791 rows
PolitiFact fake: (120, 13) | PolitiFact real: (120, 13) | BuzzFeed real: (91, 13)

PolitiFact fake columns: ['id', 'title', 'text', 'url', 'top_img', 'authors', 'source', 'publish_date', 'movies', 'images', 'canonical_link', 'meta_data', 'binary_label']


## 🏷️ Module 4 — Label Mapping & Dataset Combination

In [6]:
# ── Map LIAR labels → binary ─────────────────────────────────
LABEL_MAP = {
    'pants-fire': 0, 'false': 0, 'barely-true': 0,   # → FAKE
    'half-true':  1, 'mostly-true': 1, 'true': 1      # → REAL
}
liar_df['binary_label'] = liar_df['label'].map(LABEL_MAP)
liar_df['text']         = liar_df['statement'].astype(str)
liar_df = liar_df.dropna(subset=['binary_label'])
logger.info(f"LIAR after label mapping: {len(liar_df)} rows")

# ── Extract text from FakeNewsNet ─────────────────────────────
def extract_fnn_text(df):
    for c in ['text', 'body', 'content', 'title', 'news_title']:
        if c in df.columns:
            out = df.copy()
            out['text'] = out[c].astype(str)
            return out[['text', 'binary_label']]
    return None

fnn_parts = []
for _df, _lbl in [(pf_fake, 0), (pf_real, 1), (bf_real, 1)]:
    _d = _df.copy(); _d['binary_label'] = _lbl
    _out = extract_fnn_text(_d)
    if _out is not None:
        _out['speaker'] = 'unknown'
        fnn_parts.append(_out)

liar_clean = liar_df[['text', 'binary_label', 'speaker', 'label']].copy()
liar_clean['speaker'] = liar_clean['speaker'].fillna('unknown')
liar_clean = liar_clean.dropna()

if fnn_parts:
    fnn_df = pd.concat(fnn_parts, ignore_index=True).dropna()
    fnn_df = fnn_df[fnn_df['text'].str.len() > 30]
    fnn_df['label'] = None   # FNN has no 6-way label
    combined_df = pd.concat([liar_clean, fnn_df], ignore_index=True).sample(
        frac=1, random_state=MODEL_CONFIG['random_state']).reset_index(drop=True)
    logger.info(f"FakeNewsNet articles added: {len(fnn_df)}")
else:
    combined_df = liar_clean.copy()
    logger.warning("FakeNewsNet not found — using LIAR only")

liar_df = combined_df  # alias for downstream cells
print(f"✅ Combined dataset: {len(liar_df)} rows")
print(liar_df['binary_label'].value_counts())


13:39:45 | INFO     | LIAR after label mapping: 12791 rows
13:39:45 | INFO     | FakeNewsNet articles added: 331


✅ Combined dataset: 13122 rows
binary_label
1    7345
0    5777
Name: count, dtype: int64


## 🧠 Module 5 — Speaker Memory

Builds credibility profiles per speaker from claim history. Used as a VTI component.

In [7]:
def build_speaker_memory(df):
    """
    Build speaker credibility profiles from historical claim data.
    
    Returns dict: speaker_key → {claims, truth_rate, false_rate, pfire_rate, cred_score}
    Credibility formula: 0.60 × truth_rate + 0.25 × (1 − pfire_rate) + 0.15 × (1 − false_rate)
    """
    memory = {}
    for speaker, grp in df.groupby('speaker'):
        total      = len(grp)
        truth_rate = (grp['binary_label'] == 1).sum() / total
        false_rate = (grp['binary_label'] == 0).sum() / total
        pfire_rate = 0.0
        if 'label' in df.columns:
            pfire_rate = (grp['label'] == 'pants-fire').sum() / total
        cred = round(0.60 * truth_rate + 0.25 * (1 - pfire_rate) + 0.15 * (1 - false_rate), 4)
        memory[str(speaker).lower().strip()] = {
            'claims':     total,
            'truth_rate': round(truth_rate, 4),
            'false_rate': round(false_rate, 4),
            'pfire_rate': round(pfire_rate, 4),
            'cred_score': cred,
        }
    return memory


def display_speaker_profile(speaker):
    """Display enriched speaker profile from memory."""
    key  = str(speaker).lower().strip()
    data = SPEAKER_MEMORY.get(key)
    if not data:
        return (f"⚠️  Speaker '{speaker}' not in memory → "
                f"defaulting to credibility 50/100")
    trust_level = (
        "🟢 High"     if data['cred_score'] >= 0.70 else
        "🟡 Moderate" if data['cred_score'] >= 0.40 else
        "🔴 Low"
    )
    return {
        'Speaker':         key.title(),
        'Total Claims':    data['claims'],
        'True %':          f"{data['truth_rate']*100:.1f}%",
        'False %':         f"{data['false_rate']*100:.1f}%",
        'Pants-Fire %':    f"{data.get('pfire_rate', 0)*100:.1f}%",
        'Credibility':     f"{data['cred_score']*100:.0f} / 100",
        'Trust Level':     trust_level,
    }


SPEAKER_MEMORY = build_speaker_memory(liar_df)
logger.info(f"Speaker profiles built: {len(SPEAKER_MEMORY)}")
print(f"✅ Speaker profiles built: {len(SPEAKER_MEMORY)}")

# ── Preview: richest speaker profiles ────────────────────────
spk_df = (pd.DataFrame(SPEAKER_MEMORY).T
            .astype({'claims': int})
            .sort_values('cred_score', ascending=False))
print("\n🏆 Top 10 Most Credible Speakers:")
print(spk_df[['claims','truth_rate','false_rate','pfire_rate','cred_score']].head(10).to_string())

print("\n📋 Sample Profile Lookup:")
for _spk in ['barack-obama', 'donald-trump', 'unknown']:
    print(f"  {_spk}: {display_speaker_profile(_spk)}")


13:39:52 | INFO     | Speaker profiles built: 3310


✅ Speaker profiles built: 3310

🏆 Top 10 Most Credible Speakers:
                                  claims  truth_rate  false_rate  pfire_rate  cred_score
zoe-lofgren                            1         1.0         0.0         0.0         1.0
13th-district-gop-slate                1         1.0         0.0         0.0         1.0
zephyr-teachout                        1         1.0         0.0         0.0         1.0
yvette-mcgee-brown                     3         1.0         0.0         0.0         1.0
aarp                                   1         1.0         0.0         0.0         1.0
your-choice-colorado                   1         1.0         0.0         0.0         1.0
young-gifted-and-black-coalition       1         1.0         0.0         0.0         1.0
coast-guard                            1         1.0         0.0         0.0         1.0
william-weld                           1         1.0         0.0         0.0         1.0
wilton-gregory                         1     

## 🔬 Module 6 — Linguistic Feature Engineering (Expanded)

**Original features (20):** word_count, sent_count, avg_sent_len, sent_variance, caps_ratio, excl_density, quest_density, emo_ratio, hedge_ratio, clickbait_score, quote_ratio, num_density, title_caps, url_count, lexical_diversity, first_person, certainty_ratio, neg_framing, title_word_count, proper_sent_start

**Added features (6):** sensational_ratio, sensational_count, emo_word_count, passive_voice_ratio, uppercase_word_ratio, readability_norm


In [8]:
# ── Lexicon Sets ─────────────────────────────────────────────
EMOTIONAL   = set(['shocking','outrageous','incredible','unbelievable','horrifying',
    'amazing','terrifying','devastating','explosive','breaking','urgent',
    'exposed','truth','revealed','secret','fraud','corrupt','evil','deadly','danger'])
HEDGING     = set(['may','might','could','possibly','perhaps','allegedly','reportedly',
    'claims','suggests','appears','seems','supposedly','presumably','apparently'])
CERTAINTY   = set(['definitely','certainly','absolutely','clearly','obviously',
    'undoubtedly','proven','confirmed','fact','always','never','every','all','none'])
NEG_FRAME   = set(['fail','failure','crisis','disaster','attack','threat','violent',
    'criminal','corrupt','illegal','banned','destroyed','collapsed','worst','terrible'])
SENSATIONAL = set(['exposed','bombshell','scandal','exclusive','alert','destroys',
    'attacks','blasts','slams','fury','rage','chaos','catastrophe','epidemic',
    'invasion','conspiracy','hoax','cover-up','agenda','plot','scheme','staged',
    'suppressed','forbidden','whistleblower','insider','leaked'])
CLICKBAIT   = ["you won't believe","the truth about","what they don't want",
               'shocking','exposed','viral','jaw-dropping','mind-blowing',
               "won't believe",'they hide','must see','doctors hate']


def extract_linguistic_features(text):
    """
    Extract 26 stylometric features from text.
    All ratios are normalised per word or per sentence to be length-invariant.
    """
    text  = str(text)
    words = word_tokenize(text.lower())
    sents = sent_tokenize(text)
    alpha = [w for w in words if w.isalpha()]
    n_w   = max(len(alpha), 1)
    n_s   = max(len(sents), 1)
    slens = [len(word_tokenize(s)) for s in sents]

    # Passive voice heuristic: (was|were|is|are|been) + past-participle-ish
    passive_ct = len(re.findall(
        r'\b(was|were|been|is|are|has been|have been|had been)\s+\w+ed\b',
        text.lower()
    ))

    # Readability (Flesch Reading Ease, normalised to 0–1)
    if TEXTSTAT_AVAILABLE and len(text.split()) >= 10:
        raw_flesch      = textstat.flesch_reading_ease(text)
        readability_norm = max(0.0, min(1.0, raw_flesch / 100.0))
    else:
        readability_norm = 0.5

    # Full uppercase words (e.g. "SHOCKING", "EXPOSED")
    full_caps_ct = sum(1 for w in words if len(w) > 1 and w.isupper() and w.isalpha())

    return {
        # ── Original 20 features ──────────────────────────────
        'word_count':          n_w,
        'sent_count':          n_s,
        'avg_sent_len':        np.mean(slens) if slens else 0,
        'sent_variance':       np.var(slens)  if len(slens) > 1 else 0,
        'caps_ratio':          sum(1 for c in text if c.isupper()) / max(len(text), 1),
        'excl_density':        text.count('!') / n_s,
        'quest_density':       text.count('?') / n_s,
        'emo_ratio':           sum(1 for w in alpha if w in EMOTIONAL)  / n_w,
        'hedge_ratio':         sum(1 for w in alpha if w in HEDGING)    / n_w,
        'clickbait_score':     sum(1 for cb in CLICKBAIT if cb in text.lower()) / len(CLICKBAIT),
        'quote_ratio':         len(re.findall(r'"[^"]*"', text)) / n_s,
        'num_density':         len(re.findall(r'\b\d+\.?\d*\b', text)) / n_w,
        'title_caps':          sum(1 for w in alpha if w.istitle()) / n_w,
        'url_count':           len(re.findall(r'http\S+|www\.\S+', text)),
        'lexical_diversity':   len(set(alpha)) / n_w,
        'first_person':        sum(1 for w in words if w in ['we','our','us']) / n_w,
        'certainty_ratio':     sum(1 for w in alpha if w in CERTAINTY)  / n_w,
        'neg_framing':         sum(1 for w in alpha if w in NEG_FRAME)  / n_w,
        'title_word_count':    len(text.split()[:15]),
        'proper_sent_start':   sum(1 for s in sents
                                   if s.strip() and s.strip()[0].isupper()) / n_s,
        # ── NEW 6 features ────────────────────────────────────
        'sensational_ratio':     sum(1 for w in alpha if w in SENSATIONAL) / n_w,
        'sensational_count':     float(sum(1 for w in alpha if w in SENSATIONAL)),
        'emo_word_count':        float(sum(1 for w in alpha if w in EMOTIONAL)),
        'passive_voice_ratio':   passive_ct / n_s,
        'uppercase_word_ratio':  full_caps_ct / n_w,
        'readability_norm':      readability_norm,
    }


print("🔬 Extracting linguistic features from all samples...")
ling_feats = [extract_linguistic_features(t) for t in tqdm(liar_df['text'].tolist())]
ling_df    = pd.DataFrame(ling_feats)
LING_COLS  = ling_df.columns.tolist()
logger.info(f"Feature matrix: {ling_df.shape} ({len(LING_COLS)} features)")
print(f"\n✅ Feature matrix: {ling_df.shape}")
print(f"   Original features: 20 | New features: 6 | Total: {len(LING_COLS)}")
ling_df.describe().round(4)


🔬 Extracting linguistic features from all samples...


  0%|          | 0/13122 [00:00<?, ?it/s]

13:40:17 | INFO     | Feature matrix: (13122, 26) (26 features)



✅ Feature matrix: (13122, 26)
   Original features: 20 | New features: 6 | Total: 26


,word_count,sent_count,avg_sent_len,sent_variance,caps_ratio,excl_density,quest_density,emo_ratio,hedge_ratio,clickbait_score,...,certainty_ratio,neg_framing,title_word_count,proper_sent_start,sensational_ratio,sensational_count,emo_word_count,passive_voice_ratio,uppercase_word_ratio,readability_norm
count,13122.0000,13122.0000,13122.0000,13122.0000,13122.0000,13122.0000,13122.0000,13122.0000,13122.0000,13122.0000,...,13122.0000,13122.0000,13122.000,13122.0000,13122.0000,13122.0000,13122.0000,13122.0000,13122.0,13122.0000
mean,32.3001,1.8184,18.3942,14.0314,0.0369,0.0048,0.0046,0.0007,0.0014,0.0002,...,0.0063,0.0025,13.422,0.9684,0.0006,0.0239,0.0234,0.0578,0.0,0.5400
std,149.9504,6.1087,7.7646,134.4713,0.0266,0.0649,0.0494,0.0073,0.0092,0.0041,...,0.0213,0.0141,2.562,0.1506,0.0066,0.2311,0.2530,0.2319,0.0,0.1842
min,2.0000,1.0000,2.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,...,0.0000,0.0000,2.000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0,0.0000
25%,12.0000,1.0000,13.0000,0.0000,0.0192,0.0000,0.0000,0.0000,0.0000,0.0000,...,0.0000,0.0000,12.000,1.0000,0.0000,0.0000,0.0000,0.0000,0.0,0.4396
50%,16.0000,1.0000,17.0000,0.0000,0.0319,0.0000,0.0000,0.0000,0.0000,0.0000,...,0.0000,0.0000,15.000,1.0000,0.0000,0.0000,0.0000,0.0000,0.0,0.5287
75%,22.0000,1.0000,23.0000,0.0000,0.0476,0.0000,0.0000,0.0000,0.0000,0.0000,...,0.0000,0.0000,15.000,1.0000,0.0000,0.0000,0.0000,0.0000,0.0,0.6637
max,5459.0000,211.0000,63.0000,10574.9166,0.8214,2.5000,1.0000,0.3333,0.1667,0.1667,...,0.2500,0.2500,15.000,1.0000,0.2000,7.0000,10.0000,2.0000,0.0,1.0000


## 🌳 Module 7 — XGBoost Training

Trains on 26 linguistic features + speaker credibility score (27 total).

In [9]:
# ── Merge features + speaker credibility ────────────────────
liar_df['speaker_key'] = liar_df['speaker'].astype(str).str.lower().str.strip()
liar_df['cred_score']  = liar_df['speaker_key'].map(
    lambda k: SPEAKER_MEMORY.get(k, {}).get('cred_score', 0.5))

full_df   = pd.concat([liar_df.reset_index(drop=True), ling_df], axis=1)
full_df   = full_df.dropna(subset=['binary_label'])
FEAT_COLS = LING_COLS + ['cred_score']
logger.info(f"Training data: {full_df.shape} | Features: {len(FEAT_COLS)}")

print(f"Training data shape: {full_df.shape}")
print(f"Features used      : {len(FEAT_COLS)}")
print(f"Label distribution :\n{full_df['binary_label'].value_counts()}")

# ── Train / Test Split ───────────────────────────────────────
X = full_df[FEAT_COLS].values.astype(float)
y = full_df['binary_label'].values.astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=MODEL_CONFIG['test_size'],
    random_state=MODEL_CONFIG['random_state'],
    stratify=y
)

# ── Scale Features ───────────────────────────────────────────
scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)
logger.info("StandardScaler fitted")

# ── Train XGBoost ────────────────────────────────────────────
xgb = XGBClassifier(
    n_estimators     = MODEL_CONFIG['xgb_estimators'],
    max_depth        = MODEL_CONFIG['xgb_depth'],
    learning_rate    = MODEL_CONFIG['xgb_lr'],
    subsample        = MODEL_CONFIG['xgb_subsample'],
    colsample_bytree = MODEL_CONFIG['xgb_colsample'],
    use_label_encoder= False,
    eval_metric      = 'logloss',
    random_state     = MODEL_CONFIG['random_state'],
    n_jobs           = -1,
    verbosity        = 0
)
xgb.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
logger.info("XGBoost training complete")

# ── Evaluate ─────────────────────────────────────────────────
y_pred  = xgb.predict(X_test)
y_proba = xgb.predict_proba(X_test)[:, 1]

print("\n📊 XGBoost Results")
print("─" * 35)
print(f"  Accuracy : {accuracy_score(y_test, y_pred):.4f}")
print(f"  Precision: {precision_score(y_test, y_pred, zero_division=0):.4f}")
print(f"  Recall   : {recall_score(y_test, y_pred, zero_division=0):.4f}")
print(f"  F1 Score : {f1_score(y_test, y_pred, zero_division=0):.4f}")
print(f"  ROC-AUC  : {roc_auc_score(y_test, y_proba):.4f}")


13:40:24 | INFO     | Training data: (13122, 32) | Features: 27
13:40:24 | INFO     | StandardScaler fitted


Training data shape: (13122, 32)
Features used      : 27
Label distribution :
binary_label
1    7345
0    5777
Name: count, dtype: int64


13:40:25 | INFO     | XGBoost training complete



📊 XGBoost Results
───────────────────────────────────
  Accuracy : 0.7356
  Precision: 0.7486
  Recall   : 0.7944
  F1 Score : 0.7708
  ROC-AUC  : 0.8236


## 📊 Module 8 — Model Evaluation

In [10]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Confusion Matrix ─────────────────────────────────────────
cm   = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['Fake', 'Real'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('VeriLens AI — Confusion Matrix', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Predicted Label')
axes[0].set_ylabel('True Label')

# ── ROC Curve ────────────────────────────────────────────────
fpr, tpr, _ = roc_curve(y_test, y_proba)
auc_val     = roc_auc_score(y_test, y_proba)
axes[1].plot(fpr, tpr, color='#01696f', lw=2.5, label=f'VeriLens AUC = {auc_val:.3f}')
axes[1].plot([0, 1], [0, 1], '--', color='#bab9b4', lw=1.5, label='Random Classifier')
axes[1].fill_between(fpr, tpr, alpha=0.08, color='#01696f')
axes[1].set(xlabel='False Positive Rate', ylabel='True Positive Rate',
            xlim=[0,1], ylim=[0,1])
axes[1].set_title('VeriLens AI — ROC Curve', fontsize=13, fontweight='bold')
axes[1].legend(loc='lower right')

plt.tight_layout()
plt.savefig('eval_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: eval_plots.png")

# ── Full Classification Report ───────────────────────────────
print("\n📋 Full Classification Report:")
print("─" * 50)
print(classification_report(y_test, y_pred, target_names=['Fake','Real'], zero_division=0))

tn, fp, fn, tp = cm.ravel()
print("─" * 50)
print(f"  True Positives  (Real → Real) : {tp}")
print(f"  True Negatives  (Fake → Fake) : {tn}")
print(f"  False Positives (Fake → Real) : {fp}")
print(f"  False Negatives (Real → Fake) : {fn}")
print(f"  ROC-AUC Score                 : {auc_val:.4f}")


✅ Saved: eval_plots.png

📋 Full Classification Report:
──────────────────────────────────────────────────
              precision    recall  f1-score   support

        Fake       0.72      0.66      0.69      1156
        Real       0.75      0.79      0.77      1469

    accuracy                           0.74      2625
   macro avg       0.73      0.73      0.73      2625
weighted avg       0.73      0.74      0.73      2625

──────────────────────────────────────────────────
  True Positives  (Real → Real) : 1167
  True Negatives  (Fake → Fake) : 764
  False Positives (Fake → Real) : 392
  False Negatives (Real → Fake) : 302
  ROC-AUC Score                 : 0.8236


## 🔍 Module 9 — SHAP Explainability

SHAP TreeExplainer provides per-prediction feature attribution. Displays top positive and negative contributors.

In [12]:
print("🔍 Computing SHAP values on 200 test samples...")
explainer   = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(X_test[:200])
logger.info("SHAP explainer ready")

# ── Bar Summary Plot ──────────────────────────────────────────
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test[:200], feature_names=FEAT_COLS,
                  show=False, plot_type='bar')
plt.title('VeriLens AI — Feature Importance (SHAP)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: shap_bar.png")

# ── Dot Beeswarm Plot ─────────────────────────────────────────
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test[:200], feature_names=FEAT_COLS,
                  show=False, plot_type='dot')
plt.title('VeriLens AI — SHAP Feature Impact Direction', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('shap_dot.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: shap_dot.png")

# ── Top Positive & Negative SHAP Features ────────────────────
mean_shap = np.abs(shap_values).mean(axis=0)
mean_dir  = shap_values.mean(axis=0)  # direction: + → real, - → fake

shap_df = pd.DataFrame({
    'Feature':       FEAT_COLS,
    'Mean |SHAP|':   mean_shap.round(4),
    'Direction':     ['→ REAL' if d > 0 else '→ FAKE' for d in mean_dir]
}).sort_values('Mean |SHAP|', ascending=False).reset_index(drop=True)

print("\n🏆 Top 10 Most Important Features:")
print(shap_df.head(10).to_string(index=False))

pos_features = shap_df[shap_df['Direction']=='→ REAL'].head(5)
neg_features = shap_df[shap_df['Direction']=='→ FAKE'].head(5)
print("\n✅ Top 5 REAL indicators:", pos_features['Feature'].tolist())
print("❌ Top 5 FAKE indicators:", neg_features['Feature'].tolist())


🔍 Computing SHAP values on 200 test samples...


13:41:12 | INFO     | SHAP explainer ready


✅ Saved: shap_bar.png
✅ Saved: shap_dot.png

🏆 Top 10 Most Important Features:
          Feature  Mean |SHAP| Direction
       cred_score       1.9900    → REAL
      num_density       0.1649    → FAKE
       caps_ratio       0.1191    → FAKE
lexical_diversity       0.0915    → FAKE
 readability_norm       0.0911    → FAKE
     avg_sent_len       0.0637    → REAL
    sent_variance       0.0420    → REAL
       word_count       0.0419    → REAL
 title_word_count       0.0343    → FAKE
proper_sent_start       0.0222    → REAL

✅ Top 5 REAL indicators: ['cred_score', 'avg_sent_len', 'sent_variance', 'word_count', 'proper_sent_start']
❌ Top 5 FAKE indicators: ['num_density', 'caps_ratio', 'lexical_diversity', 'readability_norm', 'title_word_count']


## 🤖 Module 10 — DistilBERT Embeddings

Uses cosine similarity to fake/real prototype embeddings for a semantic fake-probability score.

In [13]:
DEVICE    = 'cuda' if torch.cuda.is_available() else 'cpu'
tokenizer  = AutoTokenizer.from_pretrained(MODEL_CONFIG['bert_name'])
bert_model = AutoModel.from_pretrained(MODEL_CONFIG['bert_name']).to(DEVICE)
bert_model.eval()
logger.info(f"DistilBERT loaded on: {DEVICE}")
print(f"✅ DistilBERT loaded on: {DEVICE}")


def get_bert_embedding(text):
    """Extract [CLS] embedding from DistilBERT."""
    enc = tokenizer(
        text, return_tensors='pt', truncation=True,
        max_length=MODEL_CONFIG['bert_max_len'], padding='max_length'
    ).to(DEVICE)
    with torch.no_grad():
        out = bert_model(**enc)
    return out.last_hidden_state[:, 0, :].cpu().numpy().flatten()


# ── Cache prototype embeddings (computed once) ───────────────
print("⏳ Caching prototype embeddings...")
FAKE_PROTO_EMB = get_bert_embedding(
    'false misleading manipulated fabricated unverified fake misinformation conspiracy')
REAL_PROTO_EMB = get_bert_embedding(
    'verified accurate factual credible confirmed reliable peer-reviewed true information')
print("✅ Prototype embeddings cached")


def bert_fake_probability(text):
    """Return P(fake) via softmax of cosine similarities to prototypes."""
    def cosine(a, b):
        return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))
    emb      = get_bert_embedding(str(text))
    sim_fake = cosine(emb, FAKE_PROTO_EMB)
    sim_real = cosine(emb, REAL_PROTO_EMB)
    exp_f    = math.exp(sim_fake * 5)
    exp_r    = math.exp(sim_real * 5)
    return exp_f / (exp_f + exp_r)


# ── Sanity check ─────────────────────────────────────────────
_t1 = bert_fake_probability("Scientists publish peer-reviewed vaccine study with verified results")
_t2 = bert_fake_probability("SHOCKING government conspiracy exposed — they are poisoning us!!!")
print(f"\nReal news BERT fake prob: {_t1:.3f}  (lower = more real)")
print(f"Fake news BERT fake prob: {_t2:.3f}  (higher = more fake)")


13:41:36 | INFO     | HTTP Request: HEAD https://huggingface.co/distilbert-base-uncased/resolve/main/config.json "HTTP/1.1 200 OK"
13:41:36 | WARNING  | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
13:41:36 | INFO     | HTTP Request: GET https://huggingface.co/distilbert-base-uncased/resolve/main/config.json "HTTP/1.1 200 OK"


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

13:41:36 | INFO     | HTTP Request: HEAD https://huggingface.co/distilbert-base-uncased/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
13:41:36 | INFO     | HTTP Request: GET https://huggingface.co/distilbert-base-uncased/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

13:41:36 | INFO     | HTTP Request: GET https://huggingface.co/api/models/distilbert-base-uncased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
13:41:36 | INFO     | HTTP Request: GET https://huggingface.co/api/models/distilbert/distilbert-base-uncased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
13:41:36 | INFO     | HTTP Request: GET https://huggingface.co/api/models/distilbert-base-uncased/tree/main?recursive=true&expand=false "HTTP/1.1 307 Temporary Redirect"
13:41:36 | INFO     | HTTP Request: GET https://huggingface.co/api/models/distilbert/distilbert-base-uncased/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
13:41:36 | INFO     | HTTP Request: HEAD https://huggingface.co/distilbert-base-uncased/resolve/main/vocab.txt "HTTP/1.1 200 OK"
13:41:36 | INFO     | HTTP Request: GET https://huggingface.co/distilbert-base-uncased/resolve/main/vocab.txt "HTTP/1.1 200 OK"


...

13:41:36 | INFO     | HTTP Request: HEAD https://huggingface.co/distilbert-base-uncased/resolve/main/added_tokens.json "HTTP/1.1 404 Not Found"
13:41:36 | INFO     | HTTP Request: HEAD https://huggingface.co/distilbert-base-uncased/resolve/main/special_tokens_map.json "HTTP/1.1 404 Not Found"
13:41:36 | INFO     | HTTP Request: HEAD https://huggingface.co/distilbert-base-uncased/resolve/main/chat_template.jinja "HTTP/1.1 404 Not Found"
13:41:36 | INFO     | HTTP Request: HEAD https://huggingface.co/distilbert-base-uncased/resolve/main/config.json "HTTP/1.1 200 OK"
13:41:36 | INFO     | HTTP Request: HEAD https://huggingface.co/distilbert-base-uncased/resolve/main/adapter_config.json "HTTP/1.1 404 Not Found"
13:41:37 | INFO     | HTTP Request: HEAD https://huggingface.co/distilbert-base-uncased/resolve/main/config.json "HTTP/1.1 200 OK"
13:41:37 | INFO     | HTTP Request: HEAD https://huggingface.co/distilbert-base-uncased/resolve/main/model.safetensors "HTTP/1.1 302 Found"
13:41:37 | I

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
13:41:39 | INFO     | DistilBERT loaded on: cuda


✅ DistilBERT loaded on: cuda
⏳ Caching prototype embeddings...
✅ Prototype embeddings cached

Real news BERT fake prob: 0.427  (lower = more real)
Fake news BERT fake prob: 0.554  (higher = more fake)


## 🏆 Module 11 — VeriLens Trust Index (VTI) Engine

The monolithic `compute_veriscore()` is now split into **8 focused functions**:

| Function | Returns | Weight |
|---|---|---|
| `get_model_prediction()` | XGBoost + DistilBERT blended real-probability (0–100) | 35% |
| `get_speaker_score()` | Historical speaker credibility (0–100) | 20% |
| `get_language_score()` | Language neutrality (0–100) | 15% |
| `get_clickbait_score()` | Inverse clickbait density (0–100) | 10% |
| `get_evidence_consistency_score()` | Evidence richness (0–100) | 10% |
| `get_metadata_score()` | Structural quality (0–100) | 10% |
| `calculate_trust_index()` | Weighted VTI (0–100) | — |
| `generate_explanation()` | Human-readable verdict + reasons | — |


In [14]:
# ════════════════════════════════════════════════════════════
#  VTI Component Functions  (all return 0–100, higher = more trustworthy)
# ════════════════════════════════════════════════════════════

def get_model_prediction(text, feat_sc):
    """
    Blend XGBoost P(real) and DistilBERT P(real).
    XGBoost weight 55%, DistilBERT 45%.
    """
    xgb_real  = float(xgb.predict_proba(feat_sc)[0][1])
    bert_fake = bert_fake_probability(text)
    bert_real = 1.0 - bert_fake
    model_score = 0.55 * xgb_real + 0.45 * bert_real
    return round(model_score * 100, 2)


def get_speaker_score(speaker):
    """
    Retrieve speaker credibility from memory.
    Unknown speakers default to 50 (neutral).
    Returns (score_0_to_100, raw_profile_dict).
    """
    key      = str(speaker).lower().strip()
    spk_data = SPEAKER_MEMORY.get(key, {})
    cred     = spk_data.get('cred_score', 0.5)
    return round(cred * 100, 2), spk_data


def get_language_score(lf):
    """
    Penalise emotional, sensational, overconfident, and negative language.
    Returns 0–100 (higher = more neutral).
    """
    penalties = (
        lf['emo_ratio']           * 3.0 +
        lf['certainty_ratio']     * 2.0 +
        lf['neg_framing']         * 2.0 +
        lf['caps_ratio']          * 1.5 +
        lf['sensational_ratio']   * 2.5 +
        lf['uppercase_word_ratio']* 1.0
    )
    score = max(0.0, 1.0 - penalties * 5.0)
    return round(score * 100, 2)


def get_clickbait_score(lf):
    """
    Penalise clickbait phrases and punctuation abuse.
    Returns 0–100 (higher = less clickbait = more trustworthy).
    """
    cb_penalty = (
        lf['clickbait_score'] * 4.0 +
        lf['excl_density']    * 0.3 +
        lf['quest_density']   * 0.2
    )
    score = max(0.0, 1.0 - cb_penalty)
    return round(score * 100, 2)


def get_evidence_consistency_score(lf):
    """
    Reward quoted sources, numeric data, and careful hedging.
    Returns 0–100.
    """
    evidence = (
        lf['quote_ratio']  * 2.0 +
        lf['num_density']  * 3.0 +
        lf['hedge_ratio']  * 1.0    # hedging = epistemic caution = slight positive
    )
    score = min(1.0, evidence * 2.0)
    return round(score * 100, 2)


def get_metadata_score(lf):
    """
    Reward proper sentence casing, vocabulary richness, and readability.
    Returns 0–100.
    """
    signals = (
        lf['proper_sent_start'] * 0.4 +
        lf['lexical_diversity'] * 0.3 +
        lf['readability_norm']  * 0.3
    )
    return round(min(signals, 1.0) * 100, 2)


def calculate_trust_index(components):
    """
    Compute the VeriLens Trust Index from 6 component scores.
    VTI = Σ (weight_k × score_k)   for k in VTI_WEIGHTS
    """
    vti = sum(VTI_WEIGHTS[k] * components[k] for k in VTI_WEIGHTS)
    return round(vti, 1)


def generate_explanation(vti, components, lf, spk_data, speaker):
    """
    Produce human-readable verdict, risk level, reasons, and recommendation.
    """
    if   vti >= VTI_THRESHOLDS['HIGHLY_TRUSTWORTHY']: verdict, risk = "✅ Highly Trustworthy", "Low"
    elif vti >= VTI_THRESHOLDS['MOSTLY_RELIABLE']:    verdict, risk = "🟡 Mostly Reliable",    "Low-Medium"
    elif vti >= VTI_THRESHOLDS['NEEDS_VERIFICATION']: verdict, risk = "🟠 Needs Verification", "Medium"
    elif vti >= VTI_THRESHOLDS['SUSPICIOUS']:         verdict, risk = "🔴 Suspicious",          "High"
    else:                                             verdict, risk = "❌ Likely Fake",          "Critical"

    reasons = []
    if lf['emo_ratio']           > 0.03: reasons.append("High emotional language detected")
    if lf['clickbait_score']     > 0.05: reasons.append("Clickbait patterns detected")
    if lf['caps_ratio']          > 0.08: reasons.append("Unusual character capitalisation")
    if lf['excl_density']        > 0.50: reasons.append("Excessive exclamation marks")
    if lf['certainty_ratio']     > 0.03: reasons.append("Overconfident absolute language")
    if lf['neg_framing']         > 0.03: reasons.append("Strong negative framing")
    if lf['sensational_ratio']   > 0.02: reasons.append("Sensational vocabulary detected")
    if lf['uppercase_word_ratio']> 0.03: reasons.append("Many fully-capitalised words (shouting)")
    if lf['passive_voice_ratio'] > 0.30: reasons.append("High passive voice — vague sourcing")
    if components['SPEAKER']     < 40:
        reasons.append(f"Speaker '{speaker}' has low credibility ({components['SPEAKER']:.0f}/100)")
    if lf['quote_ratio']         > 0.5:  reasons.append("Well-cited with direct quotations ✔")
    if lf['num_density']         > 0.05: reasons.append("Contains statistical / numerical evidence ✔")
    if not reasons:                      reasons.append("No strong deception signals detected ✔")

    recommendation = (
        "⚠️  HIGH RISK — Do NOT share. Verify on BBC, Reuters, Snopes, or FactCheck.org."
        if vti < 50 else
        "✔️  Appears credible. Cross-check with one trusted source before sharing."
    )
    return {'verdict': verdict, 'risk_level': risk,
            'reasons': reasons, 'recommendation': recommendation}


print("✅ VeriLens Trust Index engine functions ready")


✅ VeriLens Trust Index engine functions ready


In [15]:
def _empty_response():
    """Return a safe default when text is too short to analyse."""
    return {
        'vti':            50.0,
        'components':     {k: 50.0 for k in VTI_WEIGHTS},
        'verdict':        '⚠️ Insufficient Text',
        'risk_level':     'Unknown',
        'reasons':        ['Text too short or empty — provide more content.'],
        'recommendation': 'Please provide more text for a reliable analysis.',
        'speaker_profile':{},
        'shap_positive':  [],
        'shap_negative':  [],
        'linguistic_flags':{},
    }


def compute_vti(text, speaker='unknown', image_path=None):
    """
    Master VeriLens Trust Index computation.

    Args:
        text        : Article / claim text (str).
        speaker     : Author / speaker name (str).  Defaults to 'unknown'.
        image_path  : Path to screenshot image for OCR (str or None).

    Returns dict containing:
        vti              - VeriLens Trust Index score (0–100)
        components       - 6 component scores (dict)
        verdict          - Emoji + label string
        risk_level       - Low / Medium / High / Critical
        reasons          - List of human-readable signals
        recommendation   - Action advice string
        speaker_profile  - Raw speaker memory entry
        shap_positive    - Top SHAP features pushing toward REAL
        shap_negative    - Top SHAP features pushing toward FAKE
        linguistic_flags - Full linguistic feature dict
    """
    logger.info(f"compute_vti() | speaker={speaker} | text_len={len(str(text))}")

    # ── OCR fallback ─────────────────────────────────────────
    if image_path and (not text or len(str(text).strip()) < 20):
        logger.info(f"Running OCR on: {image_path}")
        text = extract_text_from_image(image_path)

    text = str(text).strip()
    if len(text) < 20:
        logger.warning("Text too short — returning empty response")
        return _empty_response()

    # ── Step 1: Linguistic features ──────────────────────────
    lf = extract_linguistic_features(text)

    # ── Step 2: Speaker score ────────────────────────────────
    spk_score, spk_data = get_speaker_score(speaker)

    # ── Step 3: Build feature vector for ML models ───────────
    feat    = np.array([[lf[c] for c in LING_COLS] + [spk_score / 100.0]])
    feat_sc = scaler.transform(feat.astype(float))

    # ── Step 4: Compute all 6 VTI components ─────────────────
    components = {
        'MODEL':       get_model_prediction(text, feat_sc),
        'SPEAKER':     spk_score,
        'LANGUAGE':    get_language_score(lf),
        'CLICKBAIT':   get_clickbait_score(lf),
        'CONSISTENCY': get_evidence_consistency_score(lf),
        'METADATA':    get_metadata_score(lf),
    }

    # ── Step 5: Calculate VTI ────────────────────────────────
    vti = calculate_trust_index(components)

    # ── Step 6: SHAP attribution ─────────────────────────────
    sv      = explainer.shap_values(feat_sc)[0]
    top_idx = np.argsort(np.abs(sv))[::-1][:5]
    shap_pos = [(FEAT_COLS[i], round(float(sv[i]), 4)) for i in top_idx if sv[i] > 0]
    shap_neg = [(FEAT_COLS[i], round(float(sv[i]), 4)) for i in top_idx if sv[i] < 0]

    # ── Step 7: Human-readable explanation ───────────────────
    explanation = generate_explanation(vti, components, lf, spk_data, speaker)

    logger.info(f"VTI={vti} | {explanation['verdict']} | risk={explanation['risk_level']}")

    return {
        'vti':              vti,
        'components':       components,
        'verdict':          explanation['verdict'],
        'risk_level':       explanation['risk_level'],
        'reasons':          explanation['reasons'],
        'recommendation':   explanation['recommendation'],
        'speaker_profile':  spk_data,
        'shap_positive':    shap_pos,
        'shap_negative':    shap_neg,
        'linguistic_flags': {k: round(v, 4) for k, v in lf.items()},
    }


print("✅ compute_vti() master function ready")


✅ compute_vti() master function ready


## 📷 Module 12 — OCR Module (EasyOCR)

Enables screenshot / image input. Text is extracted and passed to the VTI pipeline.

In [16]:
_ocr_reader = None  # Lazy-initialised to avoid loading on notebook startup

def get_ocr_reader():
    """Lazy-initialise EasyOCR reader (English). GPU-aware."""
    global _ocr_reader
    if _ocr_reader is None:
        try:
            import easyocr
            _ocr_reader = easyocr.Reader(['en'], gpu=torch.cuda.is_available())
            logger.info(f"EasyOCR reader initialised (gpu={torch.cuda.is_available()})")
        except ImportError:
            logger.error("easyocr not installed — run: pip install easyocr")
            raise
    return _ocr_reader


def extract_text_from_image(image_path):
    """
    Extract text from an image file using EasyOCR.

    Args:
        image_path : str — path to .jpg / .png / .webp

    Returns:
        Extracted text as a single string (empty string on failure).
    """
    if not os.path.exists(image_path):
        logger.error(f"OCR: image not found at '{image_path}'")
        return ""
    try:
        reader  = get_ocr_reader()
        results = reader.readtext(image_path, detail=0, paragraph=True)
        text    = ' '.join(results).strip()
        logger.info(f"OCR: extracted {len(text)} chars from {image_path}")
        return text
    except Exception as e:
        logger.error(f"OCR failed: {e}")
        return ""


def analyse_image(image_path, speaker='unknown'):
    """
    End-to-end: OCR an image then run VTI analysis.
    Returns the same dict as compute_vti().
    """
    logger.info(f"analyse_image() | path={image_path}")
    text = extract_text_from_image(image_path)
    if not text:
        logger.warning("OCR returned empty text")
        return _empty_response()
    print(f"📷 OCR extracted {len(text)} characters")
    print(f"   Preview: {text[:200]}...")
    return compute_vti(text, speaker=speaker)


print("✅ OCR module ready")
print("   Usage: result = analyse_image('screenshot.png', speaker='unknown')")
print("   Note : EasyOCR reader is lazy-loaded on first call (~10s on first use)")


✅ OCR module ready
   Usage: result = analyse_image('screenshot.png', speaker='unknown')
   Note : EasyOCR reader is lazy-loaded on first call (~10s on first use)


## 🚀 Module 13 — FastAPI Backend

Writes `app/backend/api.py`. Run separately with: `uvicorn app.backend.api:app --reload`

In [17]:
%%writefile app/backend/api.py
"""
VeriLens AI — FastAPI Backend
Run: uvicorn app.backend.api:app --host 0.0.0.0 --port 8000 --reload
"""
import sys, os, json, logging
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.dirname(__file__))))

from fastapi import FastAPI, HTTPException, UploadFile, File
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import Optional
import uvicorn

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s')
logger = logging.getLogger('VeriLens-API')

# ── Lazy model loading (avoids slow import on startup) ───────
_models_loaded = False
def _load_models():
    global _models_loaded, xgb, scaler, SPEAKER_MEMORY
    global FEAT_COLS, LING_COLS, explainer, VTI_WEIGHTS, VTI_THRESHOLDS
    global compute_vti, extract_text_from_image
    if not _models_loaded:
        from verilens_inference import (xgb, scaler, SPEAKER_MEMORY, FEAT_COLS,
            LING_COLS, explainer, VTI_WEIGHTS, VTI_THRESHOLDS,
            compute_vti, extract_text_from_image)
        _models_loaded = True
        logger.info("Models loaded")

# ── App ──────────────────────────────────────────────────────
app = FastAPI(
    title       = "VeriLens AI API",
    description = "Explainable Fake News Detection — VeriLens Trust Index",
    version     = "2.0.0"
)
app.add_middleware(CORSMiddleware, allow_origins=["*"],
                   allow_methods=["*"], allow_headers=["*"])

# ── Schemas ──────────────────────────────────────────────────
class PredictRequest(BaseModel):
    text:    str
    speaker: Optional[str] = "unknown"

class PredictResponse(BaseModel):
    vti:            float
    verdict:        str
    risk_level:     str
    components:     dict
    reasons:        list
    recommendation: str
    shap_positive:  list
    shap_negative:  list

# ── Endpoints ─────────────────────────────────────────────────
@app.get("/health")
def health():
    return {"status": "ok", "service": "VeriLens AI", "version": "2.0.0"}

@app.post("/predict", response_model=PredictResponse)
def predict(req: PredictRequest):
    _load_models()
    if not req.text or len(req.text.strip()) < 10:
        raise HTTPException(status_code=400, detail="Text must be at least 10 characters.")
    try:
        result = compute_vti(req.text, speaker=req.speaker or "unknown")
        logger.info(f"Prediction: VTI={result['vti']} | {result['verdict']}")
        return result
    except Exception as e:
        logger.error(f"Prediction error: {e}")
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/ocr")
async def ocr_endpoint(file: UploadFile = File(...), speaker: str = "unknown"):
    _load_models()
    import tempfile, shutil
    suffix = os.path.splitext(file.filename)[-1] or ".jpg"
    with tempfile.NamedTemporaryFile(delete=False, suffix=suffix) as tmp:
        shutil.copyfileobj(file.file, tmp)
        tmp_path = tmp.name
    try:
        text   = extract_text_from_image(tmp_path)
        result = compute_vti(text, speaker=speaker) if text else {"error": "OCR returned no text"}
        return {"extracted_text": text, "analysis": result}
    finally:
        os.unlink(tmp_path)

@app.get("/speaker/{speaker_name}")
def speaker_profile(speaker_name: str):
    _load_models()
    key  = speaker_name.lower().strip()
    data = SPEAKER_MEMORY.get(key)
    if not data:
        raise HTTPException(status_code=404, detail=f"Speaker '{speaker_name}' not in memory.")
    return {"speaker": key, **data}

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000)


Writing app/backend/api.py


## 🖥️ Module 14 — Streamlit Dashboard

Writes `app/frontend/dashboard.py`. Run with: `streamlit run app/frontend/dashboard.py`

In [18]:
%%writefile app/frontend/dashboard.py
"""
VeriLens AI — Streamlit Dashboard
Run: streamlit run app/frontend/dashboard.py
"""
import streamlit as st
import sys, os, json, joblib, numpy as np
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.dirname(__file__))))

# ── Page config ───────────────────────────────────────────────
st.set_page_config(
    page_title="VeriLens AI",
    page_icon="🔍",
    layout="wide",
    initial_sidebar_state="expanded"
)

# ── Load models (cached) ─────────────────────────────────────
@st.cache_resource
def load_models():
    from verilens_inference import (xgb, scaler, SPEAKER_MEMORY, FEAT_COLS,
        LING_COLS, explainer, VTI_WEIGHTS, compute_vti, extract_text_from_image,
        display_speaker_profile)
    return xgb, scaler, SPEAKER_MEMORY, FEAT_COLS, LING_COLS, explainer, VTI_WEIGHTS, compute_vti, extract_text_from_image, display_speaker_profile

xgb, scaler, SPEAKER_MEMORY, FEAT_COLS, LING_COLS, explainer, VTI_WEIGHTS, compute_vti, ocr_fn, spk_fn = load_models()

# ── Header ───────────────────────────────────────────────────
st.title("🔍 VeriLens AI — Fake News Detection")
st.markdown("*DistilBERT + XGBoost + SHAP | VeriLens Trust Index v2.0*")
st.divider()

# ── Sidebar ──────────────────────────────────────────────────
with st.sidebar:
    st.header("⚙️ Settings")
    speaker = st.text_input("Speaker / Author", value="unknown",
                            help="Enter speaker name for credibility lookup")
    st.divider()
    st.markdown("**VTI Component Weights**")
    for k, w in VTI_WEIGHTS.items():
        st.write(f"  {k}: **{w*100:.0f}%**")
    st.divider()
    if speaker.strip().lower() not in ('unknown', ''):
        st.subheader("👤 Speaker Profile")
        prof = spk_fn(speaker)
        if isinstance(prof, dict):
            for pk, pv in prof.items():
                st.write(f"**{pk}:** {pv}")
        else:
            st.info(prof)

# ── Input tabs ───────────────────────────────────────────────
tab1, tab2 = st.tabs(["📝 Text Input", "📷 Image / Screenshot"])

with tab1:
    text_input = st.text_area("Paste article or claim text here",
                               height=200, placeholder="Enter news article or claim...")
    col1, col2 = st.columns([1, 4])
    with col1:
        run_text = st.button("🔍 Analyse", type="primary", use_container_width=True)
    result_text = None
    if run_text and text_input.strip():
        with st.spinner("Analysing..."):
            result_text = compute_vti(text_input, speaker=speaker)

with tab2:
    uploaded = st.file_uploader("Upload screenshot", type=['png','jpg','jpeg','webp'])
    run_ocr  = st.button("🔍 OCR + Analyse", type="primary")
    result_ocr = None
    if run_ocr and uploaded:
        import tempfile, shutil
        suf = os.path.splitext(uploaded.name)[-1]
        with tempfile.NamedTemporaryFile(delete=False, suffix=suf) as tmp:
            shutil.copyfileobj(uploaded, tmp); tmp_path = tmp.name
        with st.spinner("Running OCR + VTI analysis..."):
            result_ocr = compute_vti("", speaker=speaker, image_path=tmp_path)
        os.unlink(tmp_path)

# ── Display result ────────────────────────────────────────────
result = result_text or result_ocr
if result and 'vti' in result:
    vti = result['vti']
    st.divider()
    st.subheader("📊 Analysis Results")

    col1, col2, col3, col4 = st.columns(4)
    col1.metric("VeriLens Trust Index", f"{vti}/100")
    col2.metric("Verdict", result['verdict'])
    col3.metric("Risk Level", result['risk_level'])
    col4.metric("Recommendation", "Verify ⚠️" if vti < 50 else "Credible ✔️")

    st.divider()

    # VTI Component breakdown
    st.subheader("🏆 VTI Component Breakdown")
    import plotly.graph_objects as go
    comps = result['components']
    fig = go.Figure(go.Bar(
        x=list(comps.values()), y=list(comps.keys()),
        orientation='h', marker_color='#01696f',
        text=[f"{v:.1f}" for v in comps.values()], textposition='outside'
    ))
    fig.update_layout(xaxis=dict(range=[0,100]), height=280,
                      title="Component Scores (0–100, higher = more trustworthy)",
                      margin=dict(l=20,r=20,t=40,b=20))
    st.plotly_chart(fig, use_container_width=True)

    # Explanation
    st.subheader("💡 Explanation")
    for r in result['reasons']:
        st.write(f"  • {r}")
    st.info(result['recommendation'])

    # SHAP
    col_pos, col_neg = st.columns(2)
    with col_pos:
        st.subheader("✅ REAL Indicators (SHAP)")
        for feat, val in result.get('shap_positive', []):
            st.write(f"  **{feat}**: +{val:.4f}")
    with col_neg:
        st.subheader("❌ FAKE Indicators (SHAP)")
        for feat, val in result.get('shap_negative', []):
            st.write(f"  **{feat}**: {val:.4f}")

    # Speaker profile
    if result.get('speaker_profile'):
        st.divider()
        st.subheader("👤 Speaker Profile")
        sp = result['speaker_profile']
        c1, c2, c3, c4 = st.columns(4)
        c1.metric("Total Claims", sp.get('claims','—'))
        c2.metric("True %", f"{sp.get('truth_rate',0)*100:.1f}%")
        c3.metric("False %", f"{sp.get('false_rate',0)*100:.1f}%")
        c4.metric("Credibility", f"{sp.get('cred_score',0)*100:.0f}/100")

    with st.expander("🔬 Full Linguistic Flags"):
        st.json(result.get('linguistic_flags', {}))


Writing app/frontend/dashboard.py


## 🧪 Module 15 — Demo Test Cases

In [19]:
def print_vti_report(text, speaker='unknown', label=''):
    """Pretty-print a full VTI analysis report."""
    r = compute_vti(text, speaker=speaker)
    
    print(f"{'═'*72}")
    if label: print(f"  CASE  : {label}")
    print(f"  TEXT  : {text[:80]}{'...' if len(text)>80 else ''}")
    print(f"  {'─'*68}")
    print(f"  VTI   : {r['vti']}/100  │  {r['verdict']}  │  Risk: {r['risk_level']}")
    print(f"  {'─'*68}")
    print(f"  COMPONENTS:")
    for k, v in r['components'].items():
        bar = '█' * int(v/5) + '░' * (20 - int(v/5))
        print(f"    {k:<12} [{bar}] {v:.1f}")
    print(f"  {'─'*68}")
    print(f"  REASONS:")
    for reason in r['reasons']:
        print(f"    • {reason}")
    print(f"  {'─'*68}")
    print(f"  SHAP (+): {r['shap_positive'][:3]}")
    print(f"  SHAP (-): {r['shap_negative'][:3]}")
    print(f"  {'─'*68}")
    print(f"  ADVICE : {r['recommendation']}")
    print()


test_cases = [
    ("Scientists confirm new vaccine 95% effective in peer-reviewed clinical trials",
     "unknown",       "Neutral real-sounding claim"),
    ("SHOCKING: Government secretly poisons water supply — they don't want you to know!!!",
     "unknown",       "Classic fake — emotional + clickbait"),
    ("The unemployment rate dropped by 0.3% according to Bureau of Labor Statistics",
     "barack-obama",  "Statistical claim by known speaker"),
    ("You WON'T BELIEVE what celebrities are hiding — fully EXPOSED!!!",
     "donald-trump",  "Clickbait by known speaker"),
    ("WHO reports malaria cases declining globally due to new prevention strategies",
     "unknown",       "Real health news"),
    ("The earth is flat and NASA has been lying for decades — PROOF INSIDE",
     "unknown",       "Conspiracy claim"),
]

print("\n🔍 VeriLens AI — Demo Analysis\n")
for text, spk, label in test_cases:
    print_vti_report(text, spk, label)


13:42:52 | INFO     | compute_vti() | speaker=unknown | text_len=77
13:42:52 | INFO     | VTI=72.1 | 🟡 Mostly Reliable | risk=Low-Medium
13:42:52 | INFO     | compute_vti() | speaker=unknown | text_len=83
13:42:52 | INFO     | VTI=41.5 | 🔴 Suspicious | risk=High
13:42:52 | INFO     | compute_vti() | speaker=barack-obama | text_len=77
13:42:52 | INFO     | VTI=74.2 | 🟡 Mostly Reliable | risk=Low-Medium
13:42:52 | INFO     | compute_vti() | speaker=donald-trump | text_len=64
13:42:52 | INFO     | VTI=25.0 | ❌ Likely Fake | risk=Critical
13:42:52 | INFO     | compute_vti() | speaker=unknown | text_len=77
13:42:52 | INFO     | VTI=60.2 | 🟠 Needs Verification | risk=Medium
13:42:52 | INFO     | compute_vti() | speaker=unknown | text_len=68
13:42:52 | INFO     | VTI=50.1 | 🟠 Needs Verification | risk=Medium



🔍 VeriLens AI — Demo Analysis

════════════════════════════════════════════════════════════════════════
  CASE  : Neutral real-sounding claim
  TEXT  : Scientists confirm new vaccine 95% effective in peer-reviewed clinical trials
  ────────────────────────────────────────────────────────────────────
  VTI   : 72.1/100  │  🟡 Mostly Reliable  │  Risk: Low-Medium
  ────────────────────────────────────────────────────────────────────
  COMPONENTS:
    MODEL        [██████████░░░░░░░░░░] 54.3
    SPEAKER      [██████████████░░░░░░] 72.5
    LANGUAGE     [██████████████████░░] 90.3
    CLICKBAIT    [████████████████████] 100.0
    CONSISTENCY  [███████████████░░░░░] 75.0
    METADATA     [███████████████░░░░░] 75.7
  ────────────────────────────────────────────────────────────────────
  REASONS:
    • Contains statistical / numerical evidence ✔
  ────────────────────────────────────────────────────────────────────
  SHAP (+): [('cred_score', 0.2986), ('num_density', 0.0701)]
  SHAP (-): [('

## 📈 Module 16 — VTI Distribution Analysis

Visualises VTI score distribution by true label on 300 test samples.

In [20]:
print("⏳ Computing VTI scores on 300 test samples...")
_sample   = full_df.sample(300, random_state=42).reset_index(drop=True)
_vti_rows = []
for _, row in tqdm(_sample.iterrows(), total=300):
    r = compute_vti(row['text'])
    _vti_rows.append({
        'vti':    r['vti'],
        'label':  'Real' if row['binary_label'] == 1 else 'Fake',
        'verdict': r['verdict']
    })
vti_df = pd.DataFrame(_vti_rows)

# ── Histogram ─────────────────────────────────────────────────
fig = px.histogram(
    vti_df, x='vti', color='label', barmode='overlay', nbins=30,
    color_discrete_map={'Real':'#01696f','Fake':'#a12c7b'},
    title='VeriLens Trust Index — Score Distribution by True Label',
    labels={'vti':'VeriLens Trust Index (0–100)','label':'True Label'}
)
for thresh, label_ in [(90,'Highly Trustworthy'),(70,'Mostly Reliable'),
                        (50,'Needs Verification'),(30,'Suspicious')]:
    fig.add_vline(x=thresh, line_dash='dot', line_color='gray',
                  annotation_text=label_, annotation_position='top right',
                  annotation_font_size=9)
fig.update_layout(plot_bgcolor='white', paper_bgcolor='white',
                  font=dict(size=13), height=440)
fig.show()

# ── Accuracy from VTI threshold ───────────────────────────────
vti_df['predicted'] = vti_df['vti'].apply(lambda s: 'Real' if s >= 50 else 'Fake')
acc = (vti_df['predicted'] == vti_df['label']).mean()
print(f"\n✅ VTI threshold (50) accuracy on 300 samples: {acc:.2%}")

# ── Component score comparison ────────────────────────────────
print("\n📊 Mean VTI by true label:")
print(vti_df.groupby('label')['vti'].describe().round(2))

# ── Plotly Box Plot ───────────────────────────────────────────
fig2 = px.box(vti_df, x='label', y='vti', color='label',
              color_discrete_map={'Real':'#01696f','Fake':'#a12c7b'},
              title='VTI Score Distribution — Box Plot',
              labels={'vti':'VeriLens Trust Index','label':'True Label'})
fig2.update_layout(showlegend=False, height=380,
                   plot_bgcolor='white', paper_bgcolor='white')
fig2.show()


⏳ Computing VTI scores on 300 test samples...


  0%|          | 0/300 [00:00<?, ?it/s]

13:43:27 | INFO     | compute_vti() | speaker=unknown | text_len=60
13:43:27 | INFO     | VTI=62.5 | 🟠 Needs Verification | risk=Medium
13:43:27 | INFO     | compute_vti() | speaker=unknown | text_len=87
13:43:27 | INFO     | VTI=57.6 | 🟠 Needs Verification | risk=Medium
13:43:27 | INFO     | compute_vti() | speaker=unknown | text_len=168
13:43:27 | INFO     | VTI=56.3 | 🟠 Needs Verification | risk=Medium
13:43:27 | INFO     | compute_vti() | speaker=unknown | text_len=100
13:43:27 | INFO     | VTI=72.8 | 🟡 Mostly Reliable | risk=Low-Medium
13:43:27 | INFO     | compute_vti() | speaker=unknown | text_len=43
13:43:27 | INFO     | VTI=65.2 | 🟠 Needs Verification | risk=Medium
13:43:27 | INFO     | compute_vti() | speaker=unknown | text_len=91
13:43:27 | INFO     | VTI=66.7 | 🟠 Needs Verification | risk=Medium
13:43:27 | INFO     | compute_vti() | speaker=unknown | text_len=144
13:43:27 | INFO     | VTI=72.2 | 🟡 Mostly Reliable | risk=Low-Medium
13:43:27 | INFO     | compute_vti() | speak


✅ VTI threshold (50) accuracy on 300 samples: 49.67%

📊 Mean VTI by true label:
       count   mean   std   min    25%    50%    75%   max
label                                                     
Fake   152.0  64.63  5.68  49.6  61.40  64.45  67.43  82.6
Real   148.0  66.37  6.61  50.0  63.28  66.60  70.90  82.7


## 🛡️ Module 17 — Edge Case Testing

Verifies robustness across pathological inputs.

In [21]:
print("🛡️ Running edge case tests...\n")

edge_cases = [
    ("",                         "unknown",       "Empty string"),
    ("hi",                       "unknown",       "Very short text (<5 words)"),
    ("A " * 2000,                "unknown",       "Very long repetitive text (2000 words)"),
    ("123 456 789 000 111 222",  "unknown",       "Numbers only"),
    ("!!! ??? !!! ??? !!! ???",  "unknown",       "Punctuation only"),
    ("   \n\t  ",               "unknown",       "Whitespace only"),
    ("Normal sentence here.",    "nonexistent-speaker-xyz", "Unknown speaker"),
    ("<script>alert(1)</script>","unknown",       "HTML injection attempt"),
    ("a" * 500,                  "unknown",       "500 char repeated letter"),
]

all_passed = True
for text, speaker, desc in edge_cases:
    try:
        r = compute_vti(text, speaker=speaker)
        assert isinstance(r['vti'], float), "VTI must be float"
        assert 0.0 <= r['vti'] <= 100.0,   "VTI must be in [0, 100]"
        assert 'verdict' in r,             "Must have verdict"
        assert 'reasons' in r,             "Must have reasons"
        status = "✅ PASS"
    except AssertionError as ae:
        status = f"❌ FAIL (assertion): {ae}"
        all_passed = False
    except Exception as ex:
        status = f"❌ FAIL (exception): {ex}"
        all_passed = False
    print(f"{status} | {desc}")
    print(f"           VTI={r.get('vti','?')} | Verdict={r.get('verdict','?')}")

print(f"\n{'✅ All edge case tests passed!' if all_passed else '⚠️  Some tests failed — review above.'}")


13:43:55 | INFO     | compute_vti() | speaker=unknown | text_len=0
13:43:55 | WARNING  | Text too short — returning empty response
13:43:55 | INFO     | compute_vti() | speaker=unknown | text_len=2
13:43:55 | WARNING  | Text too short — returning empty response
13:43:55 | INFO     | compute_vti() | speaker=unknown | text_len=4000
13:43:55 | INFO     | VTI=50.3 | 🟠 Needs Verification | risk=Medium
13:43:55 | INFO     | compute_vti() | speaker=unknown | text_len=23
13:43:55 | INFO     | VTI=68.6 | 🟠 Needs Verification | risk=Medium
13:43:55 | INFO     | compute_vti() | speaker=unknown | text_len=23
13:43:55 | INFO     | VTI=50.4 | 🟠 Needs Verification | risk=Medium
13:43:55 | INFO     | compute_vti() | speaker=unknown | text_len=7
13:43:55 | WARNING  | Text too short — returning empty response
13:43:55 | INFO     | compute_vti() | speaker=nonexistent-speaker-xyz | text_len=21
13:43:55 | INFO     | VTI=50.7 | 🟠 Needs Verification | risk=Medium
13:43:55 | INFO     | compute_vti() | speaker

🛡️ Running edge case tests...

✅ PASS | Empty string
           VTI=50.0 | Verdict=⚠️ Insufficient Text
✅ PASS | Very short text (<5 words)
           VTI=50.0 | Verdict=⚠️ Insufficient Text
✅ PASS | Very long repetitive text (2000 words)
           VTI=50.3 | Verdict=🟠 Needs Verification
✅ PASS | Numbers only
           VTI=68.6 | Verdict=🟠 Needs Verification
✅ PASS | Punctuation only
           VTI=50.4 | Verdict=🟠 Needs Verification
✅ PASS | Whitespace only
           VTI=50.0 | Verdict=⚠️ Insufficient Text
✅ PASS | Unknown speaker
           VTI=50.7 | Verdict=🟠 Needs Verification
✅ PASS | HTML injection attempt
           VTI=59.8 | Verdict=🟠 Needs Verification
✅ PASS | 500 char repeated letter
           VTI=62.3 | Verdict=🟠 Needs Verification

✅ All edge case tests passed!


## 💾 Module 18 — Model Persistence

In [22]:
# ── Save trained artefacts ────────────────────────────────────
joblib.dump(xgb,    MODEL_SAVE_PATH)
joblib.dump(scaler, SCALER_SAVE_PATH)
with open(SPEAKER_SAVE_PATH, 'w') as f:
    json.dump(SPEAKER_MEMORY, f, indent=2)

# ── Save metrics ──────────────────────────────────────────────
metrics = {
    'accuracy':   round(accuracy_score(y_test, y_pred), 4),
    'precision':  round(precision_score(y_test, y_pred, zero_division=0), 4),
    'recall':     round(recall_score(y_test, y_pred, zero_division=0), 4),
    'f1_score':   round(f1_score(y_test, y_pred, zero_division=0), 4),
    'roc_auc':    round(roc_auc_score(y_test, y_proba), 4),
    'train_rows': len(full_df),
    'features':   len(FEAT_COLS),
    'speakers':   len(SPEAKER_MEMORY),
    'vti_weights': VTI_WEIGHTS,
    'model_config': MODEL_CONFIG,
}
with open(METRICS_SAVE_PATH, 'w') as f:
    json.dump(metrics, f, indent=2)

print("✅ Saved:", MODEL_SAVE_PATH)
print("✅ Saved:", SCALER_SAVE_PATH)
print("✅ Saved:", SPEAKER_SAVE_PATH)
print("✅ Saved:", METRICS_SAVE_PATH)
print("\n📊 Final Metrics Summary:")
print("─" * 35)
for k, v in metrics.items():
    if isinstance(v, float):
        print(f"  {k:<15}: {v}")


✅ Saved: /kaggle/working/verilens_xgb_model.joblib
✅ Saved: /kaggle/working/verilens_scaler.joblib
✅ Saved: /kaggle/working/speaker_memory.json
✅ Saved: /kaggle/working/verilens_metrics.json

📊 Final Metrics Summary:
───────────────────────────────────
  accuracy       : 0.7356
  precision      : 0.7486
  recall         : 0.7944
  f1_score       : 0.7708
  roc_auc        : 0.8236


## 📦 Module 19 — Export Outputs

Copies all artefacts to `/kaggle/working/` and creates a zip archive.

In [24]:
shutil.make_archive(
    "/kaggle/working/verilens_ai_package",
    "zip",
    "/kaggle/working"
)

print("✅ ZIP created successfully.")

✅ ZIP created successfully.


---
## ✅ VeriLens AI — Complete Pipeline Summary

| Module | Status | Description |
|---|---|---|
| Configuration | ✅ | All weights, thresholds, paths in `VTI_WEIGHTS` / `MODEL_CONFIG` |
| Imports & Logging | ✅ | Structured logging via `logger.*` throughout |
| Dataset Loading | ✅ | LIAR + FakeNewsNet combined pipeline |
| Speaker Memory | ✅ | Credibility profiles with `display_speaker_profile()` |
| Linguistic Features | ✅ | **26 features** (+6 new: sensational, passive, readability, etc.) |
| XGBoost Training | ✅ | 300 trees, 27 features, StandardScaler |
| Model Evaluation | ✅ | Confusion matrix, ROC, classification report |
| SHAP Explainability | ✅ | Bar + beeswarm plots, top pos/neg feature display |
| DistilBERT | ✅ | Prototype-cosine semantic scoring |
| VTI Engine (Modular) | ✅ | 8 focused functions replacing monolithic `compute_veriscore()` |
| OCR Module | ✅ | EasyOCR, lazy-loaded, GPU-aware |
| FastAPI Backend | ✅ | `/predict`, `/ocr`, `/health`, `/speaker/{name}` |
| Streamlit Dashboard | ✅ | Full interactive UI with component chart, SHAP, speaker profile |
| Test Cases | ✅ | 6 demo cases with progress bar output |
| VTI Distribution | ✅ | Histogram + box plot by true label |
| Edge Case Testing | ✅ | 9 pathological inputs (empty, long, HTML, unknown speaker) |
| Model Persistence | ✅ | joblib + JSON saves with full metrics |
| Export | ✅ | All artefacts → `/kaggle/working/verilens_ai_package.zip` |

### VeriLens Trust Index Breakdown
```
Model Confidence   35% │ XGBoost P(real) × 0.55 + DistilBERT P(real) × 0.45
Speaker Reliability 20% │ Historical credibility from LIAR speaker profiles
Language Neutrality 15% │ Penalises emotional, sensational, negative framing
Clickbait Score    10% │ Clickbait phrases + punctuation abuse
Evidence Consistency 10% │ Rewards quotes, numbers, hedging
Metadata Quality   10% │ Readability, sentence casing, lexical diversity
─────────────────────
VeriLens Trust Index (0–100)
```
